In [0]:
%sql
CREATE CATALOG IF NOT EXISTS assaia_catalog

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS assaia_catalog.silver

In [0]:
table_name = "assaia_catalog.silver.dClientes"

if not spark.catalog.tableExists(table_name):

    spark.sql("""
    CREATE TABLE IF NOT EXISTS assaia_catalog.silver.dClientes (
      id               INT         NOT NULL COMMENT 'ID do cliente',
      nome_completo    STRING      NOT NULL COMMENT 'Nome completo',
      cpf              STRING      NOT NULL COMMENT 'CPF',
      genero           STRING                  COMMENT 'Gênero',
      _line            BIGINT                  COMMENT 'Identificador da linha de origem',
      _fivetran_synced TIMESTAMP               COMMENT 'Data/hora da ingestão',
      CONSTRAINT pk_dClientes PRIMARY KEY (id)
    )
    USING DELTA
    COMMENT 'Silver: dimensão clientes tratada e deduplicada'
    TBLPROPERTIES (
      'delta.autoOptimize.optimizeWrite' = 'true',
      'delta.autoOptimize.autoCompact'   = 'true'
    )
    """)

    spark.sql("""
    ALTER TABLE assaia_catalog.silver.dClientes
    ADD CONSTRAINT chk_dClientes_cpf_valido
    CHECK (length(regexp_replace(cpf, '[^0-9]', '')) = 11)
    """)

    spark.sql("""
    ALTER TABLE assaia_catalog.silver.dClientes
    ADD CONSTRAINT chk_dClientes_nome_preenchido
    CHECK (trim(nome_completo) <> '')
    """)

    spark.sql("""
    ALTER TABLE assaia_catalog.silver.dClientes
    ADD CONSTRAINT chk_dClientes_genero
    CHECK (
      genero IS NULL
      OR upper(trim(genero)) IN ('M', 'F', 'OUTRO', 'NAO INFORMADO')
    )
    """)

    print("Tabela criada.")
else:
    print("Tabela já existe.")

In [0]:
table_name = "assaia_catalog.silver.dGerentes"

if not spark.catalog.tableExists(table_name):

    spark.sql(f"""
-- =========================================================
-- 2) SILVER - dGerentes
-- =========================================================
CREATE TABLE IF NOT EXISTS assaia_catalog.silver.dGerentes (
  id                     INT         NOT NULL COMMENT 'ID do gerente',
  nome_gerente           STRING      NOT NULL COMMENT 'Nome do gerente',
  cpf                    STRING      NOT NULL COMMENT 'CPF',
  _line                  BIGINT                  COMMENT 'Identificador da linha de origem',
  _fivetran_synced       TIMESTAMP               COMMENT 'Data/hora da ingestão',
  CONSTRAINT pk_dGerentes PRIMARY KEY (id)
)
USING DELTA
COMMENT 'Silver: dimensão gerentes tratada e deduplicada'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);
""")
    spark.sql("""
ALTER TABLE assaia_catalog.silver.dGerentes
ADD CONSTRAINT chk_dGerentes_cpf_valido
CHECK (length(regexp_replace(cpf, '[^0-9]', '')) = 11);
""")
    spark.sql("""
ALTER TABLE assaia_catalog.silver.dGerentes
ADD CONSTRAINT chk_dGerentes_nome_preenchido
CHECK (trim(nome_gerente) <> '');
    """)

    print("Tabela criada.")
else:
    print("Tabela já existe.")

In [0]:
table_name = "assaia_catalog.silver.dEnderecos"

if not spark.catalog.tableExists(table_name):

    spark.sql(f"""
-- =========================================================
-- 3) SILVER - dEnderecos
-- =========================================================
CREATE TABLE IF NOT EXISTS assaia_catalog.silver.dEnderecos (
  id                     INT         NOT NULL COMMENT 'ID do endereço',
  logradouro             STRING      NOT NULL COMMENT 'Logradouro',
  bairro                 STRING      NOT NULL COMMENT 'Bairro',
  cidade                 STRING      NOT NULL COMMENT 'Cidade',
  estado                 STRING      NOT NULL COMMENT 'Estado',
  uf                     STRING      NOT NULL COMMENT 'UF',
  pais                   STRING      NOT NULL COMMENT 'País',
  _line                  BIGINT                  COMMENT 'Identificador da linha de origem',
  _fivetran_synced       TIMESTAMP               COMMENT 'Data/hora da ingestão',
  CONSTRAINT pk_dEnderecos PRIMARY KEY (id)
)
USING DELTA
COMMENT 'Silver: dimensão endereços tratada e deduplicada'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);
    """)

    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.dEnderecos
ADD CONSTRAINT chk_dEnderecos_uf_valida
CHECK (length(trim(uf)) = 2);
    """)
    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.dEnderecos
ADD CONSTRAINT chk_dEnderecos_logradouro_preenchido
CHECK (trim(logradouro) <> '');
    """)
    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.dEnderecos
ADD CONSTRAINT chk_dEnderecos_bairro_preenchido
CHECK (trim(bairro) <> '');
    """)
    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.dEnderecos
ADD CONSTRAINT chk_dEnderecos_cidade_preenchido
CHECK (trim(cidade) <> '');
    """)
    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.dEnderecos
ADD CONSTRAINT chk_dEnderecos_estado_preenchido
CHECK (trim(estado) <> '');
    """)
    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.dEnderecos
ADD CONSTRAINT chk_dEnderecos_pais_preenchido
CHECK (trim(pais) <> '');
    """)

    print("Tabela criada.")
else:
    print("Tabela já existe.")

In [0]:
table_name = "assaia_catalog.silver.dLojas"

if not spark.catalog.tableExists(table_name):

    spark.sql(f"""
-- =========================================================
-- 4) SILVER - dLojas
-- =========================================================
CREATE TABLE IF NOT EXISTS assaia_catalog.silver.dLojas (
  id                     INT         NOT NULL COMMENT 'ID da loja',
  nome_unidade           STRING      NOT NULL COMMENT 'Nome da unidade',
  endereco               STRING      NOT NULL COMMENT 'Endereço',
  id_gerente_geral       INT         NOT NULL COMMENT 'ID do gerente geral',
  _line                  BIGINT                  COMMENT 'Identificador da linha de origem',
  _fivetran_synced       TIMESTAMP               COMMENT 'Data/hora da ingestão',
  CONSTRAINT pk_dLojas PRIMARY KEY (id),
  CONSTRAINT fk_dLojas_gerente FOREIGN KEY (id_gerente_geral)
    REFERENCES assaia_catalog.silver.dGerentes (id)
)
USING DELTA
COMMENT 'Silver: dimensão lojas tratada e deduplicada'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);
    """)

    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.dLojas
ADD CONSTRAINT chk_dLojas_nome_preenchido
CHECK (trim(nome_unidade) <> '');
    """)
    
    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.dLojas
ADD CONSTRAINT chk_dLojas_endereco_preenchido
CHECK (trim(endereco) <> '');
    """)

    print("Tabela criada.")
else:
    print("Tabela já existe.")

In [0]:
table_name = "assaia_catalog.silver.dProdutos"

if not spark.catalog.tableExists(table_name):

    spark.sql(f"""
-- =========================================================
-- 5) SILVER - dProdutos
-- =========================================================
CREATE TABLE IF NOT EXISTS assaia_catalog.silver.dProdutos (
  id                     INT         NOT NULL COMMENT 'ID do produto',
  descricao              STRING      NOT NULL COMMENT 'Descrição do produto',
  marca                  STRING                  COMMENT 'Marca',
  categorias             STRING                  COMMENT 'Categoria',
  sub_categoria          STRING                  COMMENT 'Subcategoria',
  custo                  DECIMAL(18,2) NOT NULL COMMENT 'Custo',
  preco_venda            DECIMAL(18,2) NOT NULL COMMENT 'Preço de venda',
  _line                  BIGINT                  COMMENT 'Identificador da linha de origem',
  _fivetran_synced       TIMESTAMP               COMMENT 'Data/hora da ingestão',
  CONSTRAINT pk_dProdutos PRIMARY KEY (id)
)
USING DELTA
COMMENT 'Silver: dimensão produtos tratada e deduplicada'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);
    """)

    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.dProdutos
ADD CONSTRAINT chk_dProdutos_descricao_preenchida
CHECK (trim(descricao) <> '');
    """)

    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.dProdutos
ADD CONSTRAINT chk_dProdutos_custo_positivo
CHECK (custo >= 0);
    """)

    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.dProdutos
ADD CONSTRAINT chk_dProdutos_preco_positivo
CHECK (preco_venda >= 0);
    """)

    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.dProdutos
ADD CONSTRAINT chk_dProdutos_preco_maior_igual_custo
CHECK (preco_venda >= custo);

    """)

    print("Tabela criada.")
else:
    print("Tabela já existe.")

In [0]:
table_name = "assaia_catalog.silver.fVendas"

if not spark.catalog.tableExists(table_name):

    spark.sql(f"""
-- =========================================================
-- 6) SILVER - fVendas
-- =========================================================
CREATE TABLE IF NOT EXISTS assaia_catalog.silver.fVendas (
  id_cliente             INT         NOT NULL COMMENT 'ID do cliente',
  id_produto             INT         NOT NULL COMMENT 'ID do produto',
  qtd                    INT         NOT NULL COMMENT 'Quantidade',
  data_venda             TIMESTAMP   NOT NULL COMMENT 'Data da venda',
  forma_pagamento        STRING      NOT NULL COMMENT 'Forma de pagamento',
  id_loja                INT         NOT NULL COMMENT 'ID da loja',
  _line                  BIGINT                  COMMENT 'Identificador da linha de origem',
  _fivetran_synced       TIMESTAMP               COMMENT 'Data/hora da ingestão',
  CONSTRAINT fk_fVendas_cliente FOREIGN KEY (id_cliente)
    REFERENCES assaia_catalog.silver.dClientes (id),
  CONSTRAINT fk_fVendas_produto FOREIGN KEY (id_produto)
    REFERENCES assaia_catalog.silver.dProdutos (id),
  CONSTRAINT fk_fVendas_loja FOREIGN KEY (id_loja)
    REFERENCES assaia_catalog.silver.dLojas (id)
)
USING DELTA
COMMENT 'Silver: fato vendas tratado e deduplicado'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);
    """)

    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.fVendas
ADD CONSTRAINT chk_fVendas_qtd_positiva
CHECK (qtd > 0);
    """)

    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.fVendas
ADD CONSTRAINT chk_fVendas_forma_pagamento_preenchida
CHECK (trim(forma_pagamento) <> '');
    """)

    spark.sql(f"""
ALTER TABLE assaia_catalog.silver.fVendas
ADD CONSTRAINT chk_fVendas_data_valida
CHECK (data_venda IS NOT NULL);
    """)

    print("Tabela criada.")
else:
    print("Tabela já existe.")